In [134]:
import pandas as pd

## Extract

Realiza a leitura dos IDs contido no arquivo local e realiza a extração de uma planilha remota contendo os dados de todos os usuários.

In [135]:
path_ids = r'SDW2023.csv'
url_dados = r'https://github.com/NWERICKSASAKI/aprendi-na-DIO/raw/refs/heads/main/TOTVS%20-%20Fundamentos%20de%20Engenharia%20de%20Dados%20e%20Machine%20Learning/P1%20-%20Explorando%20IA%20Generativa%20em%20um%20Pipeline%20de%20ETL%20com%20Python/MOCK_DATA.csv'

df_ids = pd.read_csv(path_ids)
df_dados_brutos = pd.read_csv(url_dados)

In [136]:
df_ids

,UserID,Unnamed: 1
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN
5,7,NaN
6,11,NaN
7,171,NaN
8,555,NaN
9,1000,NaN


In [137]:
df_dados_brutos

,id,first_name,last_name,email,gender,money,allow_ads
0,1,Michaela,Sellars,msellars0@clickbank.net,Female,$5073.28,False
1,2,Hermon,Scanterbury,hscanterbury1@unesco.org,NaN,$6950.18,True
2,3,Antone,Fairbank,afairbank2@mtv.com,Male,$4886.42,False
3,4,Kurtis,Langforth,klangforth3@usda.gov,Male,$7071.09,False
4,5,Frederick,Cawthra,fcawthra4@mapy.cz,Male,$4785.95,True
...,...,...,...,...,...,...,...
995,996,Leopold,Stennings,lstenningsrn@a8.net,Male,$5559.57,False
996,997,Cam,Whitcomb,cwhitcombro@blogger.com,Male,$986.64,True
997,998,Renae,Doale,NaN,Female,$4243.49,True
998,999,Orland,Ripsher,oripsherrq@independent.co.uk,Male,$2846.05,True


## Transform

- Primeiramente seleciona os usuários da planilha com base nos IDs,
- Realiza um filtro para selecionar os clientes que autorizaram envio de mensagem promocional
- Em seguida é preenchido como gênero 'neutro' nos dados faltantes em `gender`
- Posteriomente é tratado a coluna `money` para ser interpretado como valor `float`
- Por fim é inserido uma coluna de mensagem/dica personalizada com base no gênero e valor na conta bancária do usuário.

In [138]:
ids = df_ids['UserID']
mask = df_dados_brutos['id'].isin(ids)
df_dados_selecionados = df_dados_brutos[mask]
df_dados_selecionados = df_dados_selecionados.copy()
df_dados_selecionados

,id,first_name,last_name,email,gender,money,allow_ads
0,1,Michaela,Sellars,msellars0@clickbank.net,Female,$5073.28,False
1,2,Hermon,Scanterbury,hscanterbury1@unesco.org,NaN,$6950.18,True
2,3,Antone,Fairbank,afairbank2@mtv.com,Male,$4886.42,False
3,4,Kurtis,Langforth,klangforth3@usda.gov,Male,$7071.09,False
4,5,Frederick,Cawthra,fcawthra4@mapy.cz,Male,$4785.95,True
6,7,Taryn,Eagling,teagling6@dagondesign.com,Non-binary,$6500.65,True
10,11,Barbabas,Prickett,bpricketta@1688.com,Male,$587.41,False
170,171,Izzy,Pouton,ipouton4q@wp.com,Male,$312.84,False
554,555,Kinna,Label,klabelfe@oakley.com,Female,$338.10,False
999,1000,Meghan,Planque,mplanquerr@netlog.com,Female,$767.02,True


In [139]:
mask = df_dados_selecionados['allow_ads']==True
df_dados_filtrados_por_permissao = df_dados_selecionados[mask]
df_dados_filtrados_por_permissao = df_dados_filtrados_por_permissao.copy()
df_dados_filtrados_por_permissao

,id,first_name,last_name,email,gender,money,allow_ads
1,2,Hermon,Scanterbury,hscanterbury1@unesco.org,NaN,$6950.18,True
4,5,Frederick,Cawthra,fcawthra4@mapy.cz,Male,$4785.95,True
6,7,Taryn,Eagling,teagling6@dagondesign.com,Non-binary,$6500.65,True
999,1000,Meghan,Planque,mplanquerr@netlog.com,Female,$767.02,True


In [140]:
df_corrigido = df_dados_filtrados_por_permissao.copy()
df_corrigido.loc[:,'money'] = df_corrigido['money'].str.replace('$','',regex=False)
df_corrigido.loc[:,'money'] = df_corrigido['money'].astype('float')
df_corrigido.loc[:,'gender'] = df_corrigido['gender'].fillna('Non-binary')
df = df_corrigido.copy()
df

,id,first_name,last_name,email,gender,money,allow_ads
1,2,Hermon,Scanterbury,hscanterbury1@unesco.org,Non-binary,6950.18,True
4,5,Frederick,Cawthra,fcawthra4@mapy.cz,Male,4785.95,True
6,7,Taryn,Eagling,teagling6@dagondesign.com,Non-binary,6500.65,True
999,1000,Meghan,Planque,mplanquerr@netlog.com,Female,767.02,True


In [141]:
import numpy as np

df.loc[:,'pronome'] = np.select(
    [
        df['gender'] == 'Male',
        df['gender'] == 'Female'
    ],
    [
        'Prezado',
        'Prezada'
    ],
    default = 'Olá'
)

df.loc[:,'dica'] = np.select(
    [
        df['money'] <= 1_000.00,
        (df['money'] > 1_000.00) & (df['money'] <= 5_000.00),
        df['money'] > 5_000.00
    ],
    [
        'Sabemos que está em uma fase díficil, conte conosco com empréstimos e limite especial para dar aquele impulso!',
        'Não pare agora, invista agora mesmo e garanta um futuro mais traquilo!',
        'Chegar até aqui não foi fácil, proteja seu patrimônio com nossos seguros!'
    ],
    default = 'Conte conosco para o que precisar!'
)

df['mensagem'] = df['pronome'] + ' ' + df['first_name'] + '. ' + df['dica']
df

,id,first_name,last_name,email,gender,money,allow_ads,pronome,dica,mensagem
1,2,Hermon,Scanterbury,hscanterbury1@unesco.org,Non-binary,6950.18,True,Olá,"Chegar até aqui não foi fácil, proteja seu pat...","Olá Hermon. Chegar até aqui não foi fácil, pro..."
4,5,Frederick,Cawthra,fcawthra4@mapy.cz,Male,4785.95,True,Prezado,"Não pare agora, invista agora mesmo e garanta ...","Prezado Frederick. Não pare agora, invista ago..."
6,7,Taryn,Eagling,teagling6@dagondesign.com,Non-binary,6500.65,True,Olá,"Chegar até aqui não foi fácil, proteja seu pat...","Olá Taryn. Chegar até aqui não foi fácil, prot..."
999,1000,Meghan,Planque,mplanquerr@netlog.com,Female,767.02,True,Prezada,"Sabemos que está em uma fase díficil, conte co...",Prezada Meghan. Sabemos que está em uma fase d...


## Load

Simula o envio das informações da tabela em formato JSON para a API da url fictícia.

In [ ]:
import requests

url = 'https://api.exemplo.com/clientes'

response = requests.post(
    url,
    json = df.to_dict(orient='records'),
    timeout = 10
)